# 01 - Basic checks et cleaning

But : partir du fichier Excel brut, faire les checks simples, corriger ce qui est raisonnable, puis sortir un fichier propre.

Fichier de sortie : `data_cleaned_final.xlsx`


## 1. Importer les données

In [1]:
import pandas as pd

file_path = "Cas pratique - Base.xlsx"

transactions = pd.read_excel(
    file_path,
    sheet_name="Source transactions",
    dtype={"Code client": str}
)

clients = pd.read_excel(
    file_path,
    sheet_name="Suivi par client",
    dtype={"Code client": str}
)

transactions.columns = transactions.columns.str.strip()
clients.columns = clients.columns.str.strip()

print("Transactions :", transactions.shape)
print("Clients :", clients.shape)

display(transactions.head())
display(clients.head())

Transactions : (127664, 4)
Clients : (3543, 6)


,Code client,Produit,Date,Revenue
0,041760,P1,2021-03-01,154.301667
1,006064,P1,2018-07-01,333.130000
2,012002,P2,2021-04-01,456.940000
3,000300,P3,2020-06-01,188.199000
4,002498,P1,2021-02-01,304.060000


,Code client,Activité,2018,2019,2020,2021
0,040471,Activité 1,NaN,NaN,3200.000000,3840.000000
1,038470,Activité 1,NaN,1850.95,2249.856667,375.933333
2,008919,Activité 1,1047.60,NaN,NaN,NaN
3,001458,Activité 1,NaN,723.27,1728.000000,1736.856000
4,039591,Activité 1,1826.23,1855.56,1581.480000,2520.400000


## 2. Nettoyer les formats simples

In [2]:
transactions_clean = transactions.copy()
clients_clean = clients.copy()

# Nettoyage texte simple
transactions_clean["Code client"] = transactions_clean["Code client"].astype("string").str.strip()
clients_clean["Code client"] = clients_clean["Code client"].astype("string").str.strip()

transactions_clean["Produit"] = transactions_clean["Produit"].astype("string").str.strip()
clients_clean["Activité"] = clients_clean["Activité"].astype("string").str.strip()

# Dates et revenus
transactions_clean["Date"] = pd.to_datetime(transactions_clean["Date"], errors="coerce")
transactions_clean["Revenue"] = pd.to_numeric(transactions_clean["Revenue"], errors="coerce")

print("Formats nettoyés.")

Formats nettoyés.


## 3. Corriger les IDs clients

In [3]:
def corriger_id(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    # Si Excel a ajouté .0
    if x.endswith(".0") and x[:-2].isdigit():
        x = x[:-2]

    # Si c'est un nombre, on met 6 chiffres
    if x.isdigit():
        return x.zfill(6)

    # Si c'est du texte, on garde tel quel
    return x

transactions_clean["Code client original"] = transactions_clean["Code client"]
clients_clean["Code client original"] = clients_clean["Code client"]

transactions_clean["Code client"] = transactions_clean["Code client"].apply(corriger_id)
clients_clean["Code client"] = clients_clean["Code client"].apply(corriger_id)

transactions_clean["Client_id_valide"] = transactions_clean["Code client"].astype("string").str.match(r"^\d{6}$")
clients_clean["Client_id_valide"] = clients_clean["Code client"].astype("string").str.match(r"^\d{6}$")

print("IDs corrigés quand c'était possible.")
print("IDs texte gardés car on ne peut pas les deviner.")

display(
    transactions_clean[transactions_clean["Code client"] != transactions_clean["Code client original"]]
    [["Code client original", "Code client"]]
    .drop_duplicates()
    .head(10)
)

IDs corrigés quand c'était possible.
IDs texte gardés car on ne peut pas les deviner.


,Code client original,Code client
716,12922,012922


## 4. Valeurs manquantes

In [4]:
annees = ["2018", "2019", "2020", "2021"]

print("Valeurs manquantes transactions :")
print(transactions_clean.isna().sum())

print("\nValeurs manquantes clients avant :")
print(clients_clean.isna().sum())

# Dans les colonnes de revenu annuel, vide = probablement pas de revenu cette année-là
clients_clean[annees] = clients_clean[annees].fillna(0)

# Activité vide = inconnue
clients_clean["Activité"] = clients_clean["Activité"].fillna("Activité inconnue")
clients_clean["Activité"] = clients_clean["Activité"].replace(
    ["Unknown activity", "Missing activity", ""],
    "Activité inconnue"
)

print("\nValeurs manquantes clients après :")
print(clients_clean.isna().sum())

Valeurs manquantes transactions :


Code client             0
Produit                 0
Date                    0
Revenue                 0
Code client original    0
Client_id_valide        0
dtype: int64

Valeurs manquantes clients avant :
Code client                0
Activité                   0
2018                    1130
2019                    1015
2020                     936
2021                     701
Code client original       0
Client_id_valide           0
dtype: int64

Valeurs manquantes clients après :
Code client             0
Activité                0
2018                    0
2019                    0
2020                    0
2021                    0
Code client original    0
Client_id_valide        0
dtype: int64


## 5. Doublons clients

In [5]:
clients_clean["Total_annuel"] = clients_clean[annees].sum(axis=1)

doublons_clients = clients_clean[
    clients_clean.duplicated("Code client", keep=False)
].sort_values("Code client")

print("Doublons clients trouvés :")
display(doublons_clients)

# On garde la ligne avec le plus grand revenu total
clients_clean = (
    clients_clean
    .sort_values("Total_annuel", ascending=False)
    .drop_duplicates("Code client", keep="first")
)

print("Clients après suppression des doublons :", len(clients_clean))

Doublons clients trouvés :


,Code client,Activité,2018,2019,2020,2021,Code client original,Client_id_valide,Total_annuel
896,010115,Activité 3,70914.24,72385.49,73482.24,80965.21,010115,True,297747.18
3541,010115,Activité 2,0.00,0.00,0.00,0.00,010115,True,0.00


Clients après suppression des doublons : 3542


## 6. Doublons transactions

In [6]:
nb_avant = len(transactions_clean)

transactions_clean = transactions_clean.drop_duplicates(
    ["Code client", "Produit", "Date", "Revenue"]
)

nb_apres = len(transactions_clean)

print("Transactions avant :", nb_avant)
print("Transactions après :", nb_apres)
print("Doublons supprimés :", nb_avant - nb_apres)

Transactions avant : 127664
Transactions après : 127664
Doublons supprimés : 0


## 7. Revenus négatifs

In [7]:
# Les très petits négatifs sont sûrement des arrondis
seuil = 0.01

petits_negatifs = (transactions_clean["Revenue"] < 0) & (transactions_clean["Revenue"].abs() < seuil)

transactions_clean.loc[petits_negatifs, "Revenue"] = 0

transactions_clean["Revenue_negatif"] = transactions_clean["Revenue"] < 0

print("Transactions avec revenu négatif :", transactions_clean["Revenue_negatif"].sum())
print("Total revenu négatif :", round(transactions_clean.loc[transactions_clean["Revenue_negatif"], "Revenue"].sum(), 2))

display(transactions_clean[transactions_clean["Revenue_negatif"]].head())

Transactions avec revenu négatif : 136
Total revenu négatif : -20949.92


,Code client,Produit,Date,Revenue,Code client original,Client_id_valide,Revenue_negatif
1957,000929,P3,2021-09-01,-29.881714,000929,True,True
2951,042040,P3,2021-12-01,-9.766620,042040,True,True
3516,042271,P1,2021-11-01,-116.643333,042271,True,True
3770,040135,P1,2019-05-01,-63.853750,040135,True,True
5518,039757,P1,2019-03-01,-8.976600,039757,True,True


## 8. Clients présents dans une table mais pas l'autre

In [8]:
clients_transactions = set(transactions_clean["Code client"].dropna())
clients_table = set(clients_clean["Code client"].dropna())

clients_manquants = clients_transactions - clients_table
clients_sans_transactions = clients_table - clients_transactions

print("Clients dans transactions mais pas dans clients :", clients_manquants)
print("Clients dans clients mais sans transactions :", clients_sans_transactions)

Clients dans transactions mais pas dans clients : {'012749'}
Clients dans clients mais sans transactions : {'555555'}


## 9. Ajouter les clients manquants

In [9]:
for client_id in clients_manquants:
    tmp = transactions_clean[transactions_clean["Code client"] == client_id].copy()
    tmp["Year"] = tmp["Date"].dt.year.astype(str)
    revenu_annuel = tmp.groupby("Year")["Revenue"].sum().to_dict()

    nouvelle_ligne = {
        "Code client": client_id,
        "Activité": "Activité inconnue",
        "2018": revenu_annuel.get("2018", 0),
        "2019": revenu_annuel.get("2019", 0),
        "2020": revenu_annuel.get("2020", 0),
        "2021": revenu_annuel.get("2021", 0),
        "Code client original": client_id,
        "Client_id_valide": bool(str(client_id).isdigit() and len(str(client_id)) == 6),
        "Total_annuel": sum(revenu_annuel.values())
    }

    clients_clean = pd.concat([clients_clean, pd.DataFrame([nouvelle_ligne])], ignore_index=True)

clients_clean["Client_sans_transactions"] = clients_clean["Code client"].isin(clients_sans_transactions)

print("Clients ajoutés :", len(clients_manquants))
print("Clients sans transactions :", clients_clean["Client_sans_transactions"].sum())

Clients ajoutés : 1
Clients sans transactions : 1


## 10. Créer le résumé annuel

In [10]:
transactions_clean["Year"] = transactions_clean["Date"].dt.year

revenue_annuel = (
    transactions_clean
    .groupby("Year", as_index=False)["Revenue"]
    .sum()
)

display(revenue_annuel)

,Year,Revenue
0,2018,1.418892e+07
1,2019,1.423547e+07
2,2020,1.423970e+07
3,2021,1.613468e+07


## 11. Exporter le fichier clean

In [11]:
# On garde seulement les colonnes utiles
transactions_final = transactions_clean[
    ["Code client", "Produit", "Date", "Revenue", "Client_id_valide", "Revenue_negatif"]
].copy()

clients_final = clients_clean[
    ["Code client", "Activité", "2018", "2019", "2020", "2021", "Client_id_valide", "Total_annuel", "Client_sans_transactions"]
].copy()

clients_final = clients_final.rename(columns={"Code client": "Code Client"})

with pd.ExcelWriter("data_cleaned_final.xlsx", engine="xlsxwriter") as writer:
    transactions_final.to_excel(writer, sheet_name="Carnet_transactions", index=False)
    clients_final.to_excel(writer, sheet_name="Carnet_client", index=False)
    revenue_annuel.to_excel(writer, sheet_name="Revenue_annuel", index=False)

    # Garder les IDs en texte dans Excel
    workbook = writer.book
    text_format = workbook.add_format({"num_format": "@"})
    writer.sheets["Carnet_transactions"].set_column("A:A", 15, text_format)
    writer.sheets["Carnet_client"].set_column("A:A", 15, text_format)

print("Fichier créé : data_cleaned_final.xlsx")
print("Transactions clean :", transactions_final.shape)
print("Clients clean :", clients_final.shape)

Fichier créé : data_cleaned_final.xlsx
Transactions clean : (127664, 6)
Clients clean : (3543, 9)
